# Thử nghiệm khớp đặc trưng học sâu: SIFT và SuperPoint kết hợp LightGlue (Deep Feature Matching Pilot)

### 1. Tại sao cần dùng mạng nơ-ron học sâu (Deep Learning)?
Ở thực nghiệm baseline cổ điển (classical baseline), các phương pháp truyền thống (SIFT, ORB) dựa vào công thức toán học cố định nên rất dễ ghép sai khi gặp địa hình khó (đồng cỏ lặp lại, ao hồ, ánh sáng gắt).
Notebook này sử dụng mạng nơ-ron học sâu hiện đại:
- **SuperPoint:** Mạng tích chập (CNN) được huấn luyện để phát hiện góc cạnh và kết cấu hình học bền vững.
- **LightGlue:** Mạng Transformer thông minh giúp so sánh và ghép cặp các điểm đặc trưng, tự động loại bỏ các điểm nối sai với độ chính xác vượt trội.

### 2. Mục tiêu của notebook này
- Sử dụng mô hình đã huấn luyện sẵn (Pretrained models), không cần train lại.
- Chạy thử nghiệm trực tiếp trên ảnh gốc độ phân giải cao (Full HD 1920x1080, không resize) để đánh giá:
  1. Mô hình chạy mất bao nhiêu thời gian trên phần cứng đám mây (GPU).
  2. Chất lượng bản đồ ghép ra ở độ phân giải thực tế.

Nguồn mã nguồn: cvg/LightGlue (giấy phép Apache-2.0 và giấy phép nghiên cứu phi thương mại của SuperPoint).


### 1. Thử nghiệm trên cặp ảnh (Pair-level Matching)
Chạy thử nghiệm trên 3 cặp ảnh liên tiếp đầu tiên:
- Đo thời gian xử lý của mô hình AI cho mỗi cặp ảnh.
- Vẽ các đường nối điểm trực quan để quan sát độ chuẩn xác của LightGlue.
- Lưu số liệu vào file `metrics.json`.


In [ ]:
# ==============================================================================
# CẤU HÌNH NHANH (QUICK CONFIG)
# Đổi "pilot" thành "full" để chạy toàn bộ dataset; giữ "pilot" để thử nghiệm nhanh.
# ==============================================================================
EXECUTION_MODE = "pilot"  # "pilot" hoặc "full"

import os
os.environ["UAV_EXECUTION_MODE"] = EXECUTION_MODE

from pathlib import Path
import json, time, platform, os, sys
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

# 1. Cloud environment guard (RULES.md §8: Experiments must run on Google Colab or Kaggle hosted runtimes only)
is_colab = bool(os.environ.get("COLAB_RELEASE_TAG")) and Path("/content").is_dir()
is_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) and Path("/kaggle").is_dir()
if not (is_colab or is_kaggle):
    raise RuntimeError(
        "Execution is restricted to hosted Google Colab or Kaggle runtimes (RULES.md §8). "
        "Local execution is prohibited."
    )

# 2. Automatically mount Google Drive if running on Colab for persistent storage
if is_colab:
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.ismount("/content/drive") or not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive mount failed; persistent storage unavailable.")

from datetime import datetime, timezone

# 3. Tự động cài đặt LightGlue nếu chưa có
try:
    from lightglue import LightGlue, SIFT, SuperPoint
    from lightglue.utils import load_image, rbd
except ImportError:
    print("Installing LightGlue from GitHub...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/cvg/LightGlue.git@eb42fee2d71449efb0aa5c10549752b5d75384d8"], check=True)
    from lightglue import LightGlue, SIFT, SuperPoint
    from lightglue.utils import load_image, rbd

def resolve_paths():
    """Một root dữ liệu rõ ràng; output Colab bắt buộc nằm trên Drive đã mount."""
    default_project = "/content/drive/MyDrive/mosaicking_from_uav_video_research" if is_colab else "/kaggle/working"
    project_dir = Path(os.environ.get("UAV_PROJECT_ROOT", default_project)).expanduser().resolve()
    if not project_dir.is_dir():
        raise FileNotFoundError(f"Project directory missing: {project_dir}")
    data_dir = next((project_dir / name for name in ("dataset", "data")
                     if (project_dir / name).is_dir()), None)
    if data_dir is None:
        raise FileNotFoundError(f"Expected dataset/ or data/ inside {project_dir}")
    if is_kaggle and not os.environ.get("UAV_OUTPUT_ROOT"):
        raise ValueError("Set UAV_OUTPUT_ROOT and save Kaggle version/output for persistent handoff.")
    default_output = "/content/drive/MyDrive/mosaicking_from_uav_video_research/outputs"
    output_root = Path(os.environ.get("UAV_OUTPUT_ROOT", default_output)).expanduser().resolve()
    if is_colab and not output_root.is_relative_to(Path("/content/drive").resolve()):
        raise ValueError(f"Colab output must be on mounted Google Drive: {output_root}")
    return project_dir, data_dir, output_root

REPO_ROOT, DATA_DIR, OUTPUT_ROOT = resolve_paths()
ROOT = REPO_ROOT

def _is_safe_path(base_dir: Path, target_path: Path) -> bool:
    try:
        target_path.resolve().relative_to(base_dir.resolve())
        return True
    except ValueError:
        return False

def resolve_sequence_root(path: Path) -> Path:
    """Trả về thư mục thực sự chứa thư mục rgb, hỗ trợ cả layout lồng nhau."""
    if (path / "rgb").is_dir() and any((path / "rgb").glob("*.jpg")):
        return path
    nested = path / path.name
    if (nested / "rgb").is_dir() and any((nested / "rgb").glob("*.jpg")):
        return nested
    for p in path.rglob("rgb"):
        if p.is_dir() and any(p.glob("*.jpg")):
            return p.parent
    return path

def ensure_sequence(data_dir: Path, seq_name: str) -> Path:
    """Đảm bảo thư mục sequence có ảnh; tự động giải nén an toàn nếu chỉ có archive, không giải nén lại nếu đã có."""
    target = data_dir / seq_name
    if target.is_dir():
        resolved = resolve_sequence_root(target)
        if (resolved / "rgb").is_dir() and any((resolved / "rgb").glob("*.jpg")):
            return resolved

    import tarfile, zipfile
    for ext in (".tar.gz", ".tgz", ".zip"):
        archive = data_dir / f"{seq_name}{ext}"
        if archive.is_file():
            print(f"Giải nén an toàn {archive.name}...")
            target.mkdir(parents=True, exist_ok=True)
            target_res = target.resolve()
            if ext == ".zip":
                with zipfile.ZipFile(archive, "r") as z:
                    for member in z.infolist():
                        dest = (target / member.filename).resolve()
                        if not _is_safe_path(target_res, dest):
                            raise RuntimeError(f"Unsafe zip entry path: {member.filename}")
                    z.extractall(target)
            else:
                with tarfile.open(archive, "r:*") as t:
                    if not hasattr(tarfile, "data_filter"):
                        raise RuntimeError("Safe tar extraction requires Python with tarfile.data_filter.")
                    t.extractall(target, filter="data")
            break

    resolved = resolve_sequence_root(target)
    if not ((resolved / "rgb").is_dir() and any((resolved / "rgb").glob("*.jpg"))):
        raise FileNotFoundError(f"Sequence {seq_name} not found or missing rgb/*.jpg under {target}")
    return resolved

# Chế độ chạy: pilot (3 pairs / 20 frames) hoặc full (toàn bộ pairs / frames)
EXECUTION_MODE = os.environ.get("UAV_EXECUTION_MODE", EXECUTION_MODE).lower()
if EXECUTION_MODE not in ("pilot", "full"):
    raise ValueError("UAV_EXECUTION_MODE must be pilot or full")

SEQUENCE_PATH = ensure_sequence(DATA_DIR, os.environ.get("UAV_SEQUENCE", "phantom3-village-kfs"))
files = sorted((SEQUENCE_PATH / "rgb").glob("*.jpg"), key=lambda p: float(p.stem))
if len(files) < 2:
    raise ValueError(f"Insufficient frames in {SEQUENCE_PATH}: need at least 2 frames for matching warmup, got {len(files)}.")
if EXECUTION_MODE == "pilot":
    if len(files) < 4:
        raise ValueError(f"Insufficient frames for pilot mode in {SEQUENCE_PATH}: need at least 4 frames (for 3 pairs), got {len(files)}.")
    NUM_PAIRS = 3
    MOSAIC_FRAMES = min(20, len(files))
else:
    NUM_PAIRS = len(files) - 1
    MOSAIC_FRAMES = len(files)

MAX_KEYPOINTS = 2000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Unique UTC run directory: OUTPUT_ROOT / "deep_matching_pilot" / <stamp>
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUT = OUTPUT_ROOT / "deep_matching_pilot" / RUN_STAMP
OUT.mkdir(parents=True, exist_ok=False)

torch.manual_seed(0)
cv2.setRNGSeed(0)
print(f"Mode:          {EXECUTION_MODE}")
print(f"Sequence Path: {SEQUENCE_PATH.resolve()} ({len(files)} frames)")
print(f"Output Path:   {OUT.resolve()}")
print(f"Device: {DEVICE} | Torch: {torch.__version__} | OpenCV: {cv2.__version__}")

In [ ]:
if (OUT / "completed.json").exists():
    raise RuntimeError(f"Run directory {OUT} is already marked completed. Re-run setup to allocate a fresh run directory.")

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
def check_no_projective_pole(H: np.ndarray, h: int, w: int) -> bool:
    """Verifies that the denominator w(x, y) = H[2,0]*x + H[2,1]*y + H[2,2]
    does not cross zero anywhere inside the image rectangle [0, w] x [0, h].
    Since w(x, y) is affine, it is non-zero and has consistent sign over the rectangle
    iff all four corners have strictly positive or strictly negative denominators."""
    if H is None or not np.isfinite(H).all() or np.linalg.matrix_rank(H) < 3:
        return False
    corners = np.array([
        [0.0, 0.0, 1.0],
        [float(w), 0.0, 1.0],
        [float(w), float(h), 1.0],
        [0.0, float(h), 1.0],
    ], dtype=np.float64)
    denoms = corners @ H[2, :]
    if np.any(np.abs(denoms) < 1e-7):
        return False
    return bool(np.all(denoms > 0) or np.all(denoms < 0))

@torch.inference_mode()
def safe_extract(extractor, image):
    try:
        return extractor.extract(image, resize=None)
    except (TypeError, ValueError):
        return {"keypoints": torch.empty((1, 0, 2), device=image.device),
                "descriptors": torch.empty((1, 0, 128), device=image.device)}

@torch.inference_mode()
def match_images(extractor, matcher, image0, image1):
    f0 = safe_extract(extractor, image0)
    f1 = safe_extract(extractor, image1)
    if f0["keypoints"].shape[1] == 0 or f1["keypoints"].shape[1] == 0:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
    matches = rbd(matcher({"image0": f0, "image1": f1}))["matches"]
    p0 = rbd(f0)["keypoints"][matches[:, 0]].cpu().numpy()
    p1 = rbd(f1)["keypoints"][matches[:, 1]].cpu().numpy()
    return p0, p1

records = []
for feature, extractor_type in (("sift", SIFT), ("superpoint", SuperPoint)):
    # Official SIFT wrapper includes RootSIFT, scale and orientation.
    options = {"backend": "opencv", "rootsift": True} if feature == "sift" else {}
    started = time.perf_counter()
    extractor = extractor_type(max_num_keypoints=MAX_KEYPOINTS, **options).eval().to(DEVICE)
    matcher = LightGlue(features=feature).eval().to(DEVICE)
    extractor.requires_grad_(False)
    matcher.requires_grad_(False)
    synchronize()
    setup_seconds = time.perf_counter() - started
    image0 = load_image(files[0]).to(DEVICE)
    image1 = load_image(files[1]).to(DEVICE)
    match_images(extractor, matcher, image0, image1)  # warmup on pair 0, excluded from timing
    synchronize()
    for index in range(NUM_PAIRS):
        image0 = load_image(files[index]).to(DEVICE)
        image1 = load_image(files[index + 1]).to(DEVICE)
        synchronize()
        started = time.perf_counter()
        p0, p1 = match_images(extractor, matcher, image0, image1)
        synchronize()
        match_seconds = time.perf_counter() - started
        cv2.setRNGSeed(0)
        H, mask = (cv2.findHomography(p1, p0, cv2.RANSAC, 3.0)
                   if len(p0) >= 4 else (None, None))
        good = np.zeros(len(p0), dtype=bool)
        rmse = None
        h_im, w_im = image0.shape[-2:]
        if (H is not None and np.isfinite(H).all()
                and np.linalg.matrix_rank(H) == 3 and mask is not None
                and check_no_projective_pole(H, h_im, w_im)):
            good = mask.ravel().astype(bool)
            if good.sum() >= 4:
                predicted = cv2.perspectiveTransform(p1[good].reshape(-1, 1, 2), H).reshape(-1, 2)
                rmse = float(np.sqrt(np.mean(np.sum((predicted - p0[good]) ** 2, axis=1))))
        row = dict(method=feature + "_lightglue", image0=str(files[index]),
                   image1=str(files[index+1]), matches=len(p0), inliers=int(good.sum()),
                   inlier_ratio=float(good.mean()) if len(good) else 0.0,
                   rmse_px=rmse, status="success" if rmse is not None else "failed",
                   match_seconds=match_seconds, setup_seconds=setup_seconds,
                   # match_seconds = extract + match + CPU transfer; excludes IO, setup, warmup, RANSAC, rendering
                   H_current_to_previous=H.tolist() if rmse is not None else None)
        records.append(row)
        print(row["method"], index, row["status"], row["matches"], row["inliers"], rmse, match_seconds)
        if index == 0:
            a = cv2.imread(str(files[index]))
            b = cv2.imread(str(files[index+1]))
            k0 = [cv2.KeyPoint(float(x), float(y), 1) for x,y in p0]
            k1 = [cv2.KeyPoint(float(x), float(y), 1) for x,y in p1]
            links = [cv2.DMatch(int(i), int(i), 0) for i in np.flatnonzero(good)[:60]]
            view = cv2.drawMatches(a,k0,b,k1,links,None,flags=2)
            path = OUT / (feature + "_lightglue_matches.png")
            if not cv2.imwrite(str(path), view):
                raise OSError(path)
            plt.figure(figsize=(14,5))
            plt.imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB))
            plt.title(feature + " + LightGlue: first 60 inliers")
            plt.axis("off")
            plt.show()
    del extractor, matcher
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


### 2. Dựng bản đồ hoàn chỉnh (Full Mosaic) ở độ phân giải gốc
Sau khi thử nghiệm cặp ảnh lẻ, chúng ta tiến hành nối chuỗi tuần tự 20 khung hình thành một bức ảnh bản đồ lớn:
- Sử dụng toàn bộ độ phân giải gốc 1920x1080 (không thu nhỏ).
- So sánh trực tiếp chất lượng bản đồ tạo bởi **SIFT + LightGlue** và **SuperPoint + LightGlue**.
- Hàm `build_mosaic()` sẽ dừng ngay nếu gặp một cặp ảnh bị lỗi ghép hình.


In [ ]:
if (OUT / "completed.json").exists():
    raise RuntimeError(f"Run directory {OUT} is already marked completed. Re-run setup to allocate a fresh run directory.")

import importlib.metadata, subprocess, hashlib
lg_rev = None
try:
    import lightglue
    lg_dir = Path(lightglue.__file__).resolve().parent.parent
    res = subprocess.run(["git", "rev-parse", "HEAD"], cwd=lg_dir, capture_output=True, text=True)
    if res.returncode == 0 and res.stdout.strip():
        lg_rev = res.stdout.strip()
except Exception:
    pass
if lg_rev is None:
    try:
        lg_rev = importlib.metadata.version("lightglue")
    except Exception:
        lg_rev = "installed (unknown rev)"

pair_files_needed = files[:NUM_PAIRS + 1]
pair_input_manifest = [{
    "name": p.name,
    "path": str(p.relative_to(ROOT)) if _is_safe_path(ROOT, p) else str(p),
    "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
    "size_bytes": p.stat().st_size
} for p in pair_files_needed]

report = {"device": str(DEVICE), "platform": platform.platform(),
          "torch": torch.__version__, "opencv": cv2.__version__,
          "lightglue_provenance": lg_rev,
          "mode": EXECUTION_MODE,
          "max_keypoints": MAX_KEYPOINTS, "resize": None, "ransac_px": 3.0,
          "num_pairs": NUM_PAIRS, "seed": 0,
          "inputs": pair_input_manifest,
          "scope": "pair pilot only, xem mosaic_<feature>/metrics.json cho tầng bản đồ",
          "timing": "match_seconds gồm extraction, matching, chuyển điểm về CPU; không gồm IO, setup, warmup, RANSAC, vẽ ảnh",
          "pairs": records}
with (OUT / "metrics.json").open("w", encoding="utf-8") as fh:
    json.dump(report, fh, ensure_ascii=False, indent=2, allow_nan=False)
print("Saved:", OUT / "metrics.json")

In [ ]:
def geometry(p0, p1, shape_hw=(1080, 1920)):
    H, mask = (cv2.findHomography(p1, p0, cv2.RANSAC, 3.0) if len(p0) >= 4 else (None, None))
    good = np.zeros(len(p0), bool)
    rmse = None
    if H is not None and mask is not None and np.isfinite(H).all() and np.linalg.matrix_rank(H) == 3:
        h, w = shape_hw if shape_hw is not None else (1080, 1920)
        if not check_no_projective_pole(H, h, w):
            return None, good, None
        good = mask.ravel().astype(bool)
        if good.sum() >= 4:
            predicted = cv2.perspectiveTransform(p1[good].reshape(-1, 1, 2), H).reshape(-1, 2)
            value = float(np.sqrt(np.mean(np.sum((predicted - p0[good]) ** 2, axis=1))))
            rmse = value if np.isfinite(value) else None
    return H if rmse is not None else None, good, rmse


def canvas_and_mosaic(frame_paths, homographies, max_dim=5000):
    """Warp/blend tuần tự lên canvas chung, load từng ảnh on-the-fly để giới hạn RAM."""
    corners_all = []
    shapes = []
    for path, H in zip(frame_paths, homographies):
        sample = cv2.imread(str(path))
        if sample is None:
            raise FileNotFoundError(str(path))
        h, w = sample.shape[:2]
        shapes.append((h, w))
        if not check_no_projective_pole(H, h, w):
            raise ValueError(f"Projective pole crossing canvas for {path}")
        corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], np.float32).reshape(-1, 1, 2)
        warped = cv2.perspectiveTransform(corners, H)
        if not np.isfinite(warped).all():
            raise ValueError("Canvas corners are not finite")
        corners_all.append(warped.reshape(-1, 2))
        del sample
    pts = np.vstack(corners_all)
    min_x, min_y = np.floor(pts.min(axis=0)).astype(int)
    max_x, max_y = np.ceil(pts.max(axis=0)).astype(int)
    width, height = max(1, int(max_x - min_x)), max(1, int(max_y - min_y))
    scale = min(1.0, max_dim / max(width, height))
    out_w, out_h = max(1, int(width * scale)), max(1, int(height * scale))
    S = np.diag([scale, scale, 1.0])
    T = np.array([[1.0, 0.0, -min_x], [0.0, 1.0, -min_y], [0.0, 0.0, 1.0]])

    mosaic = np.zeros((out_h, out_w, 3), np.uint8)
    covered = np.zeros((out_h, out_w), bool)
    H_to_mosaic = []
    for path, H, (h, w) in zip(frame_paths, homographies, shapes):
        img_bgr = cv2.imread(str(path))
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        del img_bgr
        full = S @ T @ H
        H_to_mosaic.append(full)
        warped = cv2.warpPerspective(img, full, (out_w, out_h))
        ones = np.ones((h, w), np.float32)
        mask = cv2.warpPerspective(ones, full, (out_w, out_h)) > 0.5
        first = mask & ~covered
        mosaic[first] = warped[first]
        blend = mask & covered
        if blend.any():
            mosaic[blend] = (0.5 * mosaic[blend].astype(np.float32)
                             + 0.5 * warped[blend].astype(np.float32)).astype(np.uint8)
        covered |= mask
        del img, warped, ones, mask
    return mosaic, H_to_mosaic, float(covered.mean()), (out_w, out_h)


def build_mosaic(matcher, method, sequence, frame_paths, limit):
    """Nối chuỗi tuần tự tới `limit` frame với bộ nhớ O(1); dừng ngay ở cặp lỗi đầu tiên."""
    chain = []
    H_accum = [np.eye(3, dtype=np.float64)]
    paths = frame_paths[:limit]

    for i in range(1, len(paths)):
        a_bgr = cv2.imread(str(paths[i - 1]))
        b_bgr = cv2.imread(str(paths[i]))
        if a_bgr is None or b_bgr is None:
            raise ValueError(f"Không đọc được ảnh: {paths[i-1]} hoặc {paths[i]}")
        a = cv2.cvtColor(a_bgr, cv2.COLOR_BGR2RGB)
        b = cv2.cvtColor(b_bgr, cv2.COLOR_BGR2RGB)
        del a_bgr, b_bgr

        started = time.perf_counter()
        p0, p1 = matcher(a, b)
        seconds = time.perf_counter() - started
        p0, p1 = np.asarray(p0, np.float32).reshape(-1, 2), np.asarray(p1, np.float32).reshape(-1, 2)
        if p0.shape != p1.shape or not np.isfinite(p0).all() or not np.isfinite(p1).all():
            raise ValueError(f"Invalid correspondences: {method}")

        cv2.setRNGSeed(0)
        H, good, rmse = geometry(p0, p1, shape_hw=b.shape[:2])

        valid = False
        if H is not None:
            H_cand = H_accum[-1] @ H
            if check_no_projective_pole(H_cand, b.shape[0], b.shape[1]):
                valid = True
                H_accum.append(H_cand)

        chain.append(dict(pair=i, status="success" if valid else "failed", matches=len(p0),
                          inliers=int(good.sum()), rmse_px=rmse, match_seconds=seconds))
        del a, b
        if not valid:
            print(f"  {method} {sequence}: cặp {i} thất bại -> dừng chuỗi ở {len(H_accum)} frame", flush=True)
            break

    stitched = len(H_accum)
    mosaic, H_to_mosaic, covered, size = canvas_and_mosaic(paths[:stitched], H_accum)
    return mosaic, H_to_mosaic, covered, size, chain, stitched

In [ ]:
if (OUT / "completed.json").exists():
    raise RuntimeError(f"Run directory {OUT} is already marked completed. Re-run setup to allocate a fresh run directory.")

MOSAIC_FRAMES = 20 if EXECUTION_MODE == "pilot" else len(files)
mosaics = {}
target_frame_paths = files[:MOSAIC_FRAMES]

mosaic_input_manifest = [{
    "name": p.name,
    "path": str(p.relative_to(ROOT)) if _is_safe_path(ROOT, p) else str(p),
    "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
    "size_bytes": p.stat().st_size
} for p in target_frame_paths]

for extractor_type, feature in ((SIFT, "sift"), (SuperPoint, "superpoint")):
    options = {"backend": "opencv", "rootsift": True} if feature == "sift" else {}
    extractor = extractor_type(max_num_keypoints=MAX_KEYPOINTS, **options).eval().to(DEVICE)
    matcher = LightGlue(features=feature).eval().to(DEVICE)
    extractor.requires_grad_(False)
    matcher.requires_grad_(False)

    def match_one(a, b):
        """a, b là numpy RGB uint8 (độ phân giải gốc) -> chuyển sang tensor cho LightGlue."""
        ta = torch.from_numpy(a).permute(2, 0, 1).float().to(DEVICE) / 255
        tb = torch.from_numpy(b).permute(2, 0, 1).float().to(DEVICE) / 255
        with torch.inference_mode():
            f0 = safe_extract(extractor, ta)
            f1 = safe_extract(extractor, tb)
            if f0["keypoints"].shape[1] == 0 or f1["keypoints"].shape[1] == 0:
                return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
            idx = rbd(matcher({"image0": f0, "image1": f1}))["matches"]
        return (rbd(f0)["keypoints"][idx[:, 0]].cpu().numpy(),
                rbd(f1)["keypoints"][idx[:, 1]].cpu().numpy())

    w0 = cv2.cvtColor(cv2.imread(str(target_frame_paths[0])), cv2.COLOR_BGR2RGB)
    w1 = cv2.cvtColor(cv2.imread(str(target_frame_paths[1])), cv2.COLOR_BGR2RGB)
    synchronize()
    match_one(w0, w1)
    synchronize()
    del w0, w1

    started = time.perf_counter()
    mosaic, H_to_mosaic, covered, size, chain, stitched = build_mosaic(
        match_one, feature, SEQUENCE_PATH.name, target_frame_paths, len(target_frame_paths))
    total = time.perf_counter() - started

    folder = OUT / f"mosaic_{feature}"
    folder.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(folder / "mosaic.png"), cv2.cvtColor(mosaic, cv2.COLOR_RGB2BGR)):
        raise OSError(folder)
    with (folder / "metrics.json").open("w", encoding="utf-8") as fh:
        json.dump({"sequence": SEQUENCE_PATH.name, "method": feature + "_lightglue",
                   "backend": str(DEVICE),
                   "resolution": "native, resize=None", "max_keypoints": MAX_KEYPOINTS,
                   "frames_requested": len(target_frame_paths), "frames_stitched": stitched,
                   "canvas_width": size[0], "canvas_height": size[1],
                   "canvas_covered_fraction": covered, "chain_total_seconds": total,
                   "inputs": mosaic_input_manifest,
                   "H_to_mosaic": [m.tolist() for m in H_to_mosaic], "chain": chain},
                  fh, indent=2, ensure_ascii=False, allow_nan=False)
    mosaics[feature] = (mosaic, size, covered, stitched, total)
    print(f"mosaic {feature:11} {stitched}/{len(target_frame_paths)} frame | {size[0]}x{size[1]} | "
          f"phủ {covered*100:.1f}% | {total:.1f}s -> {folder.name}/mosaic.png")
    del extractor, matcher
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# Cloud self-checks: projective pole rejection, blank correspondence handling
H_pole = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.01, 0.0, -1.0]], dtype=np.float64)
assert not check_no_projective_pole(H_pole, 1080, 1920), "Projective pole check failed in pilot!"
assert check_no_projective_pole(np.eye(3), 1080, 1920), "Identity check failed in pilot!"
g_H, g_inliers, g_rmse = geometry(np.empty((0, 2)), np.empty((0, 2)))
assert g_H is None and not g_inliers.any() and g_rmse is None, "Empty geometry check failed!"
print("PASS: Cloud self-checks passed in pilot.")

# Write completed.json LAST after all methods and self-checks succeed
completed_path = OUT / "completed.json"
completed_payload = {
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "status": "success",
    "experiment": "deep_matching_pilot",
    "sequence": SEQUENCE_PATH.name,
    "mode": EXECUTION_MODE,
    "frames_requested": len(target_frame_paths),
    "methods": ["sift_lightglue", "superpoint_lightglue"],
    "inputs": mosaic_input_manifest,
    "provenance": {
        "torch": torch.__version__,
        "opencv": cv2.__version__,
        "lightglue_provenance": lg_rev,
        "device": str(DEVICE),
        "platform": platform.platform()
    }
}
with completed_path.open("w", encoding="utf-8") as f:
    json.dump(completed_payload, f, indent=2, allow_nan=False)
print(f"Completed run marker written: {completed_path}")

In [ ]:
# Hiển thị cả 2 bản đồ để nhìn bằng mắt
fig, axes = plt.subplots(1, len(mosaics), figsize=(8 * len(mosaics), 7))
for ax, (feature, (mosaic, size, covered, stitched, total)) in zip(np.ravel(axes), mosaics.items()):
    ax.imshow(mosaic)
    ax.set_title(f"{feature} + LightGlue (độ phân giải gốc)\n"
                 f"{size[0]}x{size[1]} | phủ {covered*100:.0f}% | {stitched} frame | {total:.1f}s", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()
